# FinDER × RDF vs LPG: five evaluation tracks (fully embedded)

We build the same FinDER knowledge graph twice — once as **LPG** (labeled property graph backed by LanceDB) and once as **RDF / OWL** (backed by [owlready2](https://owlready2.readthedocs.io/), a Python OWL store that loads FIBO ontologies natively). Both run inside this Jupyter container — no graph server required.

**Why owlready2 for the RDF side?**

- FIBO is published as OWL ontologies; owlready2 loads them directly
- It exposes OWL classes and individuals as Python classes (so a node label maps to a class, an edge type maps to an `ObjectProperty`)
- It supports SPARQL via projection to rdflib for the comparison queries
- It can run an OWL reasoner (`sync_reasoner`) — the kind of inference LPG simply cannot perform
- This matches how the rest of seocho already uses owlready2 (offline ontology governance only — see `seocho/ontology_governance.py`)

**Five evaluation tracks:**

| Track | What it checks |
|---|---|
| 1. Golden Standard | Class/relationship overlap against the reference FIBO module set |
| 2. Data-Driven | Fraction of corpus-extracted entities the ontology has a class for |
| 3. Application/Task | FinDER QA correctness via Cypher-style retrieval (LPG) and SPARQL (RDF) |
| 4. User-based | Likert form — qualitative; emitted as a CSV scaffold |
| 5. Structure-based | Network metrics (LPG-only; RDF skipped because triples aren't first-class edges) |

**Bonus track: OWL reasoning.** After the five evaluations we run `sync_reasoner` over the OWL store and compare the inferred-triple count to the asserted count. This is the LPG-can't-do-this slot.

**Requirements.** All embedded — `owlready2`, `rdflib`, `lancedb`, `networkx`. No external services. The OWL reasoner step needs Java (HermiT); the notebook reports gracefully if no JVM is found.

## Setup

In [ ]:
import json
import os
import sys
import time
from pathlib import Path

from dotenv import load_dotenv

ROOT = Path.cwd().parent if (Path.cwd().name == "examples") else Path.cwd()
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")

if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY is required.")

FINDER_PATH = os.environ.get(
    "FINDER_PATH",
    str(ROOT / "examples" / "datasets" / "finder_tutorial_subset.json"),
)
WORKDIR = ROOT / ".seocho" / "finder_rdf_vs_lpg"
WORKDIR.mkdir(parents=True, exist_ok=True)
print(f"FinDER:   {FINDER_PATH}")
print(f"Workdir:  {WORKDIR}")

In [ ]:
from seocho.benchmarking import load_finder_cases
from seocho.index.pipeline import IndexingPipeline
from seocho.store.llm import create_llm_backend

from examples.datasets.fibo_modules.compose import compose_modules
from examples.lance_graph_store import LanceGraphStore
from examples.owlready_graph_store import OwlreadyGraphStore
from examples.lpg_metrics import compute_lpg_structure_metrics
from examples.rdf_lpg_comparison import (
    corpus_coverage,
    golden_standard_overlap,
    task_track_aggregate,
    write_user_eval_template,
)

cases = load_finder_cases(FINDER_PATH)
llm = create_llm_backend(provider="openai", model="gpt-4o-mini")
print(f"Loaded {len(cases)} FinDER cases")

## Build LPG and RDF/OWL ontologies

Both sides use the **full** FIBO module composition (BE + FBC + SEC + FND + IND). The only difference is `graph_model` and the FIBO namespace passed to owlready2.

In [ ]:
lpg_ontology = compose_modules(["be", "fbc", "sec", "fnd", "ind"])
lpg_ontology.graph_model = "lpg"

rdf_ontology = compose_modules(["be", "fbc", "sec", "fnd", "ind"])
rdf_ontology.graph_model = "rdf"
rdf_ontology.namespace = "https://spec.edmcouncil.org/fibo/"

lpg_store = LanceGraphStore(uri=str(WORKDIR / "lpg.lance"))
rdf_store = OwlreadyGraphStore(
    ontology=rdf_ontology,
    namespace=rdf_ontology.namespace,
    store_path=str(WORKDIR / "rdf.sqlite3"),
)
print(f"LPG schema:  {lpg_store.get_schema()}")
print(f"RDF schema:  {rdf_store.get_schema()}")

## Index FinDER documents into both stores

In [ ]:
captured = {"lpg": {"nodes": [], "rels": []}, "rdf": {"nodes": [], "rels": []}}

def make_pipeline(ontology, store, workspace_id, bucket):
    def cap(nodes, rels):
        bucket["nodes"].extend(nodes)
        bucket["rels"].extend(rels)
        return nodes, rels
    return IndexingPipeline(
        ontology=ontology,
        graph_store=store,
        llm=llm,
        workspace_id=workspace_id,
        on_after_extract=cap,
    )

for name, ontology, store, ws in (
    ("lpg", lpg_ontology, lpg_store, "finder_lpg"),
    ("rdf", rdf_ontology, rdf_store, "finder_rdf"),
):
    print(f"== Building {name} graph ==")
    pipeline = make_pipeline(ontology, store, ws, captured[name])
    for case in cases:
        try:
            pipeline.index(case.text, metadata={"case_id": case.case_id})
        except Exception as exc:
            print(f"  index failure on {case.case_id}: {exc}")
    print(f"  captured {len(captured[name]['nodes'])} nodes, {len(captured[name]['rels'])} rels")

## Track 1 — Golden Standard overlap

In [ ]:
fibo_reference_classes = list(lpg_ontology.nodes.keys())

def constructed_classes(bucket):
    return {n.get("label", "") for n in bucket["nodes"] if n.get("label")}

track1 = {
    "lpg": golden_standard_overlap(constructed_classes(captured["lpg"]), fibo_reference_classes),
    "rdf": golden_standard_overlap(constructed_classes(captured["rdf"]), fibo_reference_classes),
}
for path, m in track1.items():
    print(f"  {path}: jaccard={m['jaccard']:.2f}  precision={m['precision']:.2f}  recall={m['recall']:.2f}")

## Track 2 — Data-Driven coverage

In [ ]:
alias_map = {
    label: list(node_def.aliases or [])
    for label, node_def in lpg_ontology.nodes.items()
}
extracted_class_names = lambda bucket: [str(n.get("label", "")) for n in bucket["nodes"] if n.get("label")]
track2 = {
    "lpg": corpus_coverage(extracted_class_names(captured["lpg"]), alias_map),
    "rdf": corpus_coverage(extracted_class_names(captured["rdf"]), alias_map),
}
for path, m in track2.items():
    print(f"  {path}: classified={m['classified_count']}/{m['total_distinct_entities']} coverage={m['coverage']:.2f}")

## Track 3 — Application/Task (FinDER QA)

We answer the FinDER questions through each store with a small retriever appropriate for that representation:
- **LPG:** keyword-anchored neighbor expansion on `LanceGraphStore` (same recipe as Tutorial 1).
- **RDF:** SPARQL `SELECT` against the OWL store (via `OwlreadyGraphStore.sparql`).

Both retrieved contexts feed the same OpenAI synthesizer so the comparison is apples-to-apples on the answer side.

In [ ]:
import re

STOPWORDS = {"what", "where", "who", "when", "how", "the", "a", "an", "is", "was", "were", "of", "in", "to", "for", "on", "and", "did", "during"}
TOKEN_RE = re.compile(r"[A-Za-z][A-Za-z0-9_.&\-]+")

def question_keywords(question: str) -> list[str]:
    return [t for t in TOKEN_RE.findall(question) if t.lower() not in STOPWORDS]

def lpg_context(question: str) -> str:
    facts: list[str] = []
    seen: set[str] = set()
    for kw in question_keywords(question):
        for seed in lpg_store.keyword_retrieve(kw, limit=3):
            if seed["id"] in seen:
                continue
            seen.add(seed["id"])
            facts.append(f"{seed['label']}({seed['name']}) properties={seed['properties']}")
            for hop in lpg_store.neighbors(seed["id"], limit=5):
                arrow = "->" if hop["direction"] == "out" else "<-"
                facts.append(
                    f"{seed['name']} {arrow}[{hop['edge_type']}]{arrow} {hop['neighbor_label']}({hop['neighbor_name']})"
                )
    return "\n".join(facts)

def rdf_context(question: str) -> str:
    rows: list[str] = []
    for kw in question_keywords(question):
        # Retrieve individuals whose name or label matches the keyword.
        sparql_query = f'''
PREFIX fibo: <{rdf_ontology.namespace}>
SELECT ?ind ?cls ?annot
WHERE {{
  ?ind a ?cls .
  OPTIONAL {{ ?ind fibo:has_property ?annot . }}
  FILTER(
    CONTAINS(LCASE(STR(?ind)), LCASE("{kw.lower()}")) ||
    CONTAINS(LCASE(COALESCE(STR(?annot), "")), LCASE("{kw.lower()}"))
  )
}}
LIMIT 5
'''
        try:
            for row in rdf_store.sparql(sparql_query):
                rows.append(f"{row.get('cls','')}: {row.get('ind','')} | {row.get('annot','')}")
        except Exception:
            continue
    return "\n".join(rows)

def answer(context: str, question: str) -> dict:
    if not context:
        return {"answer": "(no graph evidence)", "executed_ok": False}
    prompt = (
        "Answer using only the graph evidence. Be concise.\n\n"
        f"Graph evidence:\n{context}\n\nQuestion: {question}\nAnswer:"
    )
    return {"answer": llm.complete(prompt).strip(), "executed_ok": True}

lpg_answers, rdf_answers = [], []
for case in cases:
    lpg_a = answer(lpg_context(case.question), case.question)
    rdf_a = answer(rdf_context(case.question), case.question)
    lpg_answers.append({"case_id": case.case_id, "question": case.question, "answer": lpg_a["answer"], "expected": case.expected_answer, "executed_ok": lpg_a["executed_ok"]})
    rdf_answers.append({"case_id": case.case_id, "question": case.question, "answer": rdf_a["answer"], "expected": case.expected_answer, "executed_ok": rdf_a["executed_ok"]})

track3 = {"lpg": task_track_aggregate(lpg_answers), "rdf": task_track_aggregate(rdf_answers)}
for path, m in track3.items():
    print(f"  {path}: contains_match={m['contains_match_rate']:.2f}  exec_success={m['exec_success_rate']:.2f}")

## Track 4 — User-based (CSV scaffold)

In [ ]:
user_eval_questions = [
    {
        "question": lpg_a["question"],
        "lpg_answer": lpg_a["answer"],
        "rdf_answer": rdf_a["answer"],
    }
    for lpg_a, rdf_a in zip(lpg_answers, rdf_answers)
]
scaffold_path = WORKDIR / "user_eval_template.csv"
write_user_eval_template(scaffold_path, questions=user_eval_questions, reviewers=5)
print(f"Wrote {scaffold_path}")
print("Distribute to reviewers, fill the Likert columns, then aggregate by mean score per path.")

## Track 5 — Structure-based (LPG only)

Network metrics on the LanceDB-backed property graph. RDF is skipped because owlready2 stores triples — without a property-graph projection, degree distribution and clustering aren't first-class on the RDF side.

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
from collections import Counter

# LanceGraphStore doesn't speak Cypher, so we read its tables directly
# rather than going through compute_lpg_structure_metrics (which is
# Cypher-based for Neo4j).  Same metrics, different transport.
node_rows = lpg_store._table_to_rows(lpg_store._nodes)
edge_rows = lpg_store._table_to_rows(lpg_store._edges)

g = nx.DiGraph()
for n in node_rows:
    g.add_node(n["id"], label=n.get("label"))
for e in edge_rows:
    if e.get("source") and e.get("target"):
        g.add_edge(e["source"], e["target"], type=e.get("type"))

deg_hist = Counter(d for _, d in g.degree())
fig, ax = plt.subplots(1, 1, figsize=(7, 4))
if deg_hist:
    ax.bar(list(deg_hist.keys()), list(deg_hist.values()))
ax.set_title("LPG degree distribution")
ax.set_xlabel("degree")
ax.set_ylabel("node count")
plt.tight_layout()
plt.show()

track5 = {
    "lpg": {
        "node_count": g.number_of_nodes(),
        "edge_count": g.number_of_edges(),
        "density": float(nx.density(g)) if g.number_of_nodes() > 1 else 0.0,
        "average_clustering": float(nx.average_clustering(g.to_undirected())) if g.number_of_nodes() > 0 else 0.0,
        "weakly_connected_components": int(nx.number_weakly_connected_components(g)) if g.number_of_nodes() > 0 else 0,
        "pagerank_top": [
            {"node": k, "score": float(v)}
            for k, v in (sorted(nx.pagerank(g).items(), key=lambda kv: kv[1], reverse=True)[:5] if g.number_of_nodes() > 0 else [])
        ],
    },
    "rdf": {"note": "skipped — RDF needs property-graph projection for native network metrics."},
}
print(json.dumps(track5, indent=2))

## Bonus — OWL reasoning (LPG can't do this)

Run the HermiT reasoner over the OWL store and compare asserted vs inferred triple counts. Skipped gracefully if no JVM is present (HermiT is Java).

In [ ]:
rdf_graph = rdf_store._world.as_rdflib_graph()
asserted = len(rdf_graph)
result = rdf_store.run_reasoner()
rdf_graph_after = rdf_store._world.as_rdflib_graph()
inferred = len(rdf_graph_after) - asserted
print(f"reasoner ok: {result.get('ok')} (skipped reason: {result.get('error', '-')})")
print(f"asserted triples: {asserted}")
print(f"inferred triples: {inferred} (added by sync_reasoner)")
print("\nA few inferred type assertions:")
for s, p, o in list(rdf_graph_after.triples((None, None, None)))[:6]:
    print(f"  {s} {p} {o}")

## Scoreboard

In [ ]:
scoreboard = {
    "track1_golden_standard": track1,
    "track2_data_driven": {
        "lpg": {"coverage": track2["lpg"]["coverage"]},
        "rdf": {"coverage": track2["rdf"]["coverage"]},
    },
    "track3_application_task": track3,
    "track4_user_based": {"note": f"populate {scaffold_path.name} and aggregate."},
    "track5_structure_based": track5,
    "bonus_owl_reasoning": {
        "asserted_triples": asserted,
        "inferred_triples": inferred,
        "reasoner_ok": result.get('ok', False),
    },
}
(WORKDIR / "scoreboard.json").write_text(json.dumps(scoreboard, indent=2))
print(json.dumps(scoreboard, indent=2))

## When to choose which

- **LPG (LanceDB)** — pick when you want network analytics (centrality, communities, paths), schema-light experimentation, or Cypher ergonomics. Track 5 is essentially free and the bonus reasoning track is irrelevant.
- **RDF / OWL (owlready2)** — pick when portability matters (FIBO, OWL, SHACL natively), when answers must cite IRIs to ground in a public ontology, or when downstream consumers expect SPARQL. Track 1 should consistently win on RDF if the ontology is faithfully imported, and the bonus reasoning track is RDF-only.

Both stores in this tutorial are embedded — no servers, no network ports. To scale up, swap `LanceGraphStore` for `Neo4jGraphStore` and `OwlreadyGraphStore` for the same `Neo4jGraphStore` configured with `Ontology.graph_model="rdf"` (which writes via neosemantics). The seocho APIs upstream of the stores stay identical.

Cleanup — both stores live under `.seocho/finder_rdf_vs_lpg/`:
```
rm -rf .seocho/finder_rdf_vs_lpg
```